# Setup — add the "holidays" table to datateam_capacity_v2

Creates the city-holidays table as a new layer **inside the existing** datateam_capacity_v2 feature service — it becomes the next layer/table index, alongside team_members / absences / time_entries / allocations.

**This notebook WRITES to AGOL** (unlike the audit notebooks). Run it once. You must be the **owner or an admin** of the datateam_capacity_v2 service.

After it runs, copy the printed table URL into the app's ARCGIS_CONFIG as `holidaysUrl` (src/agol.js).

**Schema rationale**
- One row per observed holiday date (no recurring rules — observed dates shift, so we store specifics each year).
- No `hours` field: holidays subtract the person's full scheduled hours for that day-of-week (so a 9/80 Monday loses 9h, a 5/8 Monday loses 8h, an RDO loses 0h).
- Date Only field — no time component needed.
- Index on `holiday_date` for fast "is today a holiday?" lookups.

In [ ]:
from arcgis.gis import GIS
from arcgis.features import FeatureLayerCollection

gis = GIS("home")
print(f"Signed in as {gis.users.me.username} @ {gis.url}")

SERVICE_URL = "https://services3.arcgis.com/9coHY2fvuFjG9HQX/ArcGIS/rest/services/datateam_capacity_v2/FeatureServer"
flc = FeatureLayerCollection(SERVICE_URL, gis)

existing_layers = [l.properties.name for l in flc.layers]
existing_tables = [t.properties.name for t in flc.tables]
print("Layers:", existing_layers)
print("Tables:", existing_tables)

## Table definition

Two fields beyond OBJECTID: `holiday_date` (Date Only) and `name` (Text). An index on `holiday_date` keeps the per-day "is this a holiday?" check cheap. No editor tracking is needed — admins manage via the in-app Settings page; the audit trail is implicit in AGOL's edit history.

In [ ]:
TABLE_NAME = "holidays"

table_def = {
    "type": "Table",
    "name": TABLE_NAME,
    "description": "City holidays for the Analytics Project Tracker. One row per observed date. Subtracted from project capacity for any person scheduled to work that day.",
    "hasAttachments": False,
    "hasM": False,
    "hasZ": False,
    "objectIdField": "OBJECTID",
    "globalIdField": "",
    "supportsAdvancedQueries": True,
    "allowGeometryUpdates": False,
    "capabilities": "Create,Delete,Query,Update,Editing",
    "fields": [
        {"name": "OBJECTID",     "type": "esriFieldTypeOID",       "alias": "OBJECTID",     "nullable": False, "editable": False},
        {"name": "holiday_date", "type": "esriFieldTypeDateOnly",  "alias": "Holiday Date", "nullable": False, "editable": True},
        {"name": "name",         "type": "esriFieldTypeString",    "alias": "Name",         "length": 255, "nullable": False, "editable": True}
    ],
    "indexes": [
        {"name": "hol_date_idx", "fields": "holiday_date", "isAscending": True, "isUnique": False, "description": "date lookup"}
    ]
}

print("Defined", len(table_def["fields"]), "fields for table:", TABLE_NAME)

## Create it (the write step)

Guarded so re-running is safe — if a table named `holidays` already exists, it does nothing.

In [ ]:
if TABLE_NAME in existing_tables:
    print(f"'{TABLE_NAME}' already exists in this service — nothing to do.")
else:
    result = flc.manager.add_to_definition({"tables": [table_def]})
    print("add_to_definition result:", result)

## Verify + get the URL

Re-reads the service and prints the new table's REST URL and fields. Copy the URL into `src/agol.js` as `holidaysUrl`.

In [ ]:
flc2 = FeatureLayerCollection(SERVICE_URL, gis)  # re-read so the new table shows
found = None
for t in flc2.tables:
    if t.properties.name == TABLE_NAME:
        found = t
        break

if not found:
    print("Could not find the new table — check the add_to_definition result above.")
else:
    print("holidaysUrl:", found.url)
    print()
    print("Fields:")
    for f in found.properties.fields:
        print(f"  {f['name']:14s} {f['type']}")
    print()
    print("Sanity query (should be 0 rows):", found.query(where="1=1", return_count_only=True))

## (Optional) Seed FY26 city holidays

Pre-populates the table with the 12 standard City of Tucson observed holidays for calendar year 2026. **Skip this cell** if you'd rather enter them via the in-app Settings UI once the code change ships, or if you want to verify the table structure first.

Dates below reflect Tucson's official observed-date convention (federal-style: when a holiday falls on a weekend, observed Friday or Monday). Spot-check against the City HR calendar before running — observed dates occasionally differ year-to-year for floating holidays.

In [ ]:
# Run this cell to seed; comment out to skip.
SEED_2026_HOLIDAYS = [
    ("2026-01-01", "New Year's Day"),
    ("2026-01-19", "Martin Luther King Jr. Day"),
    ("2026-02-16", "Presidents' Day"),
    ("2026-05-25", "Memorial Day"),
    ("2026-06-19", "Juneteenth"),
    ("2026-07-03", "Independence Day (observed)"),
    ("2026-09-07", "Labor Day"),
    ("2026-11-11", "Veterans Day"),
    ("2026-11-26", "Thanksgiving Day"),
    ("2026-11-27", "Day after Thanksgiving"),
    ("2026-12-24", "Christmas Eve"),
    ("2026-12-25", "Christmas Day"),
]

if not found:
    print("Table not found — run the previous cell first.")
else:
    # Skip dates that already exist (idempotent re-run).
    existing_rows = found.query(where="1=1", out_fields="holiday_date,name").features
    existing_dates = set()
    for f in existing_rows:
        d = f.attributes.get("holiday_date")
        if isinstance(d, str):
            existing_dates.add(d[:10])
        elif isinstance(d, (int, float)):
            from datetime import datetime, timezone
            existing_dates.add(datetime.fromtimestamp(d / 1000, tz=timezone.utc).strftime("%Y-%m-%d"))

    to_add = [{"attributes": {"holiday_date": d, "name": n}} for (d, n) in SEED_2026_HOLIDAYS if d not in existing_dates]
    if not to_add:
        print("All seed holidays already present — nothing to add.")
    else:
        result = found.edit_features(adds=to_add)
        added = sum(1 for r in (result.get("addResults") or []) if r.get("success"))
        failed = sum(1 for r in (result.get("addResults") or []) if not r.get("success"))
        print(f"Added {added} holidays, {failed} failed.")
        if failed:
            print("Errors:", [r for r in result.get("addResults", []) if not r.get("success")])

## Next step

In `src/agol.js`, add to ARCGIS_CONFIG (next to `allocationsUrl`):

    holidaysUrl:       '<paste the URL printed above>',

Then the app's holiday-aware capacity / Settings UI can load against it.

To remove the table later (cleanup):

    flc.manager.delete_from_definition({"tables": [{"name": "holidays"}]})

Or delete it from the service's Data tab in AGOL.